##Read the three Bronze master tables

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

CATALOG = "capstone_8hospital"
BRONZE = f"{CATALOG}.bronze_schema"
SILVER = f"{CATALOG}.silver_schema"
spark.sql(f"create schema if not exists {SILVER}")
patients_bronze = spark.table(f"{BRONZE}.bronze_patients")
departments_bronze = spark.table(f"{BRONZE}.bronze_departments")
beds_bronze = spark.table(f"{BRONZE}.bronze_beds")

print("Patients:", patients_bronze.count())
print("Departments:", departments_bronze.count())
print("Beds:", beds_bronze.count())

Patients: 183
Departments: 192
Beds: 201


##Transform Patient Master

In [0]:
patients_silver = (
    patients_bronze
    .select(
        F.trim("patient_id").alias("patient_id"),
        F.trim("synthetic_patient_code").alias("synthetic_patient_code"),
        F.trim("age_band").alias("age_band"),
        F.upper(F.trim("sex")).alias("sex"),
        F.trim("city").alias("city"),
        F.to_date("registration_date").alias("registration_date"),
        F.upper(F.trim("consent_flag")).alias("consent_flag"),
        F.upper(F.trim("insurance_type")).alias("insurance_type"),
        F.upper(F.trim("patient_status")).alias("patient_status"),
        F.trim("primary_language").alias("primary_language")
    )
)

##Transform Department Master

In [0]:
departments_silver = (
    departments_bronze
    .select(
        F.trim("department_id").alias("department_id"),
        F.trim("department_name").alias("department_name"),
        F.trim("specialty").alias("specialty"),
        F.trim("hospital_site").alias("hospital_site"),
        F.col("floor").cast("int").alias("floor"),
        F.upper(F.trim("service_level")).alias("service_level"),
        F.upper(F.trim("active_flag")).alias("active_flag"),
        F.trim("head_clinician_id").alias("head_clinician_id"),
        F.col("capacity_beds").cast("int").alias("capacity_beds"),
        F.trim("region").alias("region")
    )
)

##Transform Bed Master

In [0]:
beds_silver = (
    beds_bronze
    .select(
        F.trim("bed_id").alias("bed_id"),
        F.trim("department_id").alias("department_id"),
        F.trim("ward").alias("ward"),
        F.upper(F.trim("bed_type")).alias("bed_type"),
        F.upper(F.trim("isolation_capable")).alias("isolation_capable"),
        F.upper(F.trim("bed_status")).alias("bed_status"),
        F.col("floor").cast("int").alias("floor"),
        F.trim("hospital_site").alias("hospital_site"),
        F.upper(F.trim("active_flag")).alias("active_flag"),
        F.to_date("effective_date").alias("effective_date")
    )
)

##Validate primary keys

In [0]:
def validate_master(df, key):
    return (
        df.withColumn(
            "dq_reason",
            F.when(F.col(key).isNull() | (F.trim(F.col(key)) == ""), f"MISSING_{key.upper()}")
             .otherwise("")
        )
    )

patients_checked = validate_master(patients_silver, "patient_id")
departments_checked = validate_master(departments_silver, "department_id")
beds_checked = validate_master(beds_silver, "bed_id")

patients_quarantine = patients_checked.filter(F.col("dq_reason") != "")
departments_quarantine = departments_checked.filter(F.col("dq_reason") != "")
beds_quarantine = beds_checked.filter(F.col("dq_reason") != "")

patients_silver = patients_checked.filter(F.col("dq_reason") == "").drop("dq_reason").dropDuplicates(["patient_id"])
departments_silver = departments_checked.filter(F.col("dq_reason") == "").drop("dq_reason").dropDuplicates(["department_id"])
beds_silver = beds_checked.filter(F.col("dq_reason") == "").drop("dq_reason").dropDuplicates(["bed_id"])

##Save the three Silver master tables

In [0]:
for table_name, dataframe in {
    "silver_patients": patients_silver,
    "silver_departments": departments_silver,
    "silver_beds": beds_silver
}.items():
    (dataframe.write.format("delta").mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{SILVER}.{table_name}"))

master_quarantine = (
    patients_quarantine.withColumn("source_entity", F.lit("PATIENTS"))
    .unionByName(departments_quarantine.withColumn("source_entity", F.lit("DEPARTMENTS")), allowMissingColumns=True)
    .unionByName(beds_quarantine.withColumn("source_entity", F.lit("BEDS")), allowMissingColumns=True)
)

(master_quarantine.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{SILVER}.silver_master_quarantine"))

##Silver Master Validation

In [0]:
for table_name in ["silver_patients", "silver_departments", "silver_beds", "silver_master_quarantine"]:
    print(table_name, spark.table(f"{SILVER}.{table_name}").count())

silver_patients 183
silver_departments 192
silver_beds 201
silver_master_quarantine 0
